# Module 13 — Notebook 4 Solutions: Mini-Project — Full Error Analysis

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_keys, check_length

# Load data
data_path = Path("../../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

print("Setup complete. Total outputs:", len(outputs))

## Step 1 — Solution: Classifier V1 and Confusion Matrix

In [ ]:
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    return any(trigger in response for trigger in TRIGGERS_V1)

predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

confusion_v1 = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
for pred, label in zip(predictions_v1, ground_truth):
    if pred and label:
        confusion_v1['tp'] += 1
    elif pred and not label:
        confusion_v1['fp'] += 1
    elif not pred and label:
        confusion_v1['fn'] += 1
    else:
        confusion_v1['tn'] += 1

print("Confusion matrix:", confusion_v1)

In [ ]:
check_equal(confusion_v1['tp'], 5, "TP = 5")
check_equal(confusion_v1['fp'], 0, "FP = 0")
check_equal(confusion_v1['fn'], 2, "FN = 2")
check_equal(confusion_v1['tn'], 13, "TN = 13")

## Step 2 — Solution: Extract False Negative Examples

In [ ]:
fn_examples = [
    record
    for record, pred, label in zip(outputs, predictions_v1, ground_truth)
    if not pred and label
]

fn_ids = [r['id'] for r in fn_examples]

print(f"False negatives: {len(fn_examples)}")
for r in fn_examples:
    print(f"  {r['id']} ({r['model']}): {r['response']!r}")

In [ ]:
check_contains(fn_ids, 'out_011', "fn_ids contains out_011")
check_contains(fn_ids, 'out_015', "fn_ids contains out_015")

## Step 3 — Solution: Slice Analysis

In [ ]:
models = sorted(set(r['model'] for r in outputs))
per_model_fn_rate = {}

for model in models:
    model_triples = [
        (r, pred, label)
        for r, pred, label in zip(outputs, predictions_v1, ground_truth)
        if r['model'] == model
    ]
    fn_count = sum(1 for _, pred, label in model_triples if not pred and label)
    total = len(model_triples)
    per_model_fn_rate[model] = round(fn_count / total, 4)

worst_model = max(per_model_fn_rate, key=per_model_fn_rate.get)

print("Per-model FN rate:", per_model_fn_rate)
print("Worst-performing model:", worst_model)

In [ ]:
check_equal(worst_model, 'model-b-v1', "Worst model is model-b-v1")

## Step 4 — Solution: Propose Improvements

In [ ]:
improvements = {
    'add_keyword': 'Keywords are not the right tool here — both FNs contain no suspicious phrasing. A semantic or factual-check approach is needed rather than adding more keywords.',
    'add_factual_check': 'Use a retrieval-augmented fact-checker: for responses that make factual claims, query a knowledge base and flag responses whose claims contradict it. This would catch out_011 (Great Wall claim) and out_015 (2+2=5).',
    'notes': 'Both FNs come exclusively from model-b-v1, suggesting a model-specific evaluation strategy. Consider training a secondary classifier on model-b-v1 outputs to detect subtle factual errors that keyword matching cannot catch.',
}

for key, value in improvements.items():
    print(f"  {key}: {value}")

In [ ]:
check_keys(improvements, ['add_keyword', 'add_factual_check', 'notes'], "improvements has correct keys")
check_type(improvements['add_keyword'], str, "add_keyword is a string")
check_type(improvements['add_factual_check'], str, "add_factual_check is a string")
check_type(improvements['notes'], str, "notes is a string")
check_equal(len(improvements['notes']) > 10, True, "notes is more than 10 characters")